# Tool-Augmented Reasoning: ReAct and Beyond

Pure language model reasoning is bounded by the model's parametric knowledge and arithmetic capability. Tool augmentation breaks this ceiling: connecting models to calculators, search engines, code interpreters, and databases enables exact computation and up-to-date information.

## ReAct: Reason + Act

ReAct (Yao et al. 2022) interleaves reasoning traces with tool-use actions. The model alternates between:
- **Thought**: internal reasoning about what to do next
- **Action**: a tool call (search, lookup, calculate)
- **Observation**: the result returned by the tool

This pattern naturally handles failures: if an action returns an error or unexpected result, the model can reason about it and try a different approach. ReAct outperformed pure CoT on knowledge-intensive QA tasks (HotpotQA, FEVER) where factual grounding mattered.

ReAct became the foundational pattern for modern agent frameworks (LangChain, LlamaIndex, Claude tool use).

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import re

@dataclass
class ReActStep:
    step_type: str  # 'thought', 'action', 'observation'
    content: str

@dataclass
class ReActTrace:
    problem: str
    steps: list = field(default_factory=list)
    final_answer: str = ''
    n_tool_calls: int = 0

class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name: str, fn: Callable, description: str):
        self.tools[name] = {'fn': fn, 'description': description}

    def call(self, name: str, args: str) -> str:
        if name not in self.tools:
            return f'Error: tool {name} not found. Available: {list(self.tools.keys())}'
        try:
            return str(self.tools[name]['fn'](args))
        except Exception as e:
            return f'Error calling {name}: {e}'

    def descriptions(self) -> str:
        return '\n'.join(f'- {k}: {v["description"]}' for k, v in self.tools.items())

class ReActAgent:
    def __init__(self, model_fn: Callable, tools: ToolRegistry, max_steps: int = 10):
        self.model = model_fn
        self.tools = tools
        self.max_steps = max_steps

    def _parse_action(self, text: str) -> Optional[tuple]:
        m = re.search(r'Action:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if m:
            return m.group(1), m.group(2).strip()
        return None

    def run(self, problem: str) -> ReActTrace:
        trace = ReActTrace(problem=problem)
        context = f'Tools available:\n{self.tools.descriptions()}\n\nProblem: {problem}\n'
        for _ in range(self.max_steps):
            response = self.model(context)
            if 'Final Answer:' in response:
                answer = re.search(r'Final Answer:\s*(.+)', response)
                trace.final_answer = answer.group(1).strip() if answer else response
                trace.steps.append(ReActStep('answer', trace.final_answer))
                break
            action = self._parse_action(response)
            if action:
                tool_name, tool_args = action
                thought = response.split('Action:')[0].replace('Thought:', '').strip()
                trace.steps.append(ReActStep('thought', thought[:80]))
                result = self.tools.call(tool_name, tool_args)
                trace.steps.append(ReActStep('action', f'{tool_name}[{tool_args[:40]}]'))
                trace.steps.append(ReActStep('observation', result[:100]))
                trace.n_tool_calls += 1
                context += f'Thought: {thought}\nAction: {tool_name}[{tool_args}]\nObservation: {result}\n'
            else:
                trace.steps.append(ReActStep('thought', response[:80]))
                context += f'Thought: {response}\n'
        return trace

# Tools
registry = ToolRegistry()
registry.register('calculator', lambda expr: eval(expr, {'__builtins__': {}}, {}),
                   'Evaluate arithmetic: calculator[2+2] returns 4')
registry.register('search', lambda q: f'Search result for "{q}": found relevant information.',
                   'Web search: search[query] returns relevant snippets')
registry.register('lookup', lambda k: {'pi': '3.14159', 'e': '2.71828'}.get(k, 'Not found'),
                   'Constant lookup: lookup[pi] returns 3.14159')

# Mock model that uses tools
step_counter = [0]
def react_model(context):
    step_counter[0] += 1
    if step_counter[0] == 1:
        return 'Thought: I need to compute 15% of 80.\nAction: calculator[80 * 0.15]'
    if step_counter[0] == 2:
        return 'Thought: The calculator returned 12.0, which is correct.\nFinal Answer: 12.0'
    return 'Final Answer: unknown'

agent = ReActAgent(react_model, registry, max_steps=5)
trace = agent.run('What is 15% of 80?')
print(f'Problem: {trace.problem}')
print(f'Tool calls: {trace.n_tool_calls}')
for step in trace.steps:
    print(f'  [{step.step_type}] {step.content}')
print(f'Answer: {trace.final_answer}')

## Tool Selection and Routing

With many tools available, the model must decide when to use which tool:

**Calculator**: any arithmetic beyond simple mental math. Models make arithmetic errors at surprisingly low complexity (e.g. 7-digit multiplication).

**Code interpreter**: complex calculations, data manipulation, simulation. More general than a calculator but slower and higher risk.

**Search/retrieval**: factual questions about current events, domain-specific data, or any claim that requires grounding.

**Structured lookup**: when the answer is a specific fact in a database (stock prices, chemical properties, country populations).

Tool routing errors — calling the wrong tool, providing malformed arguments, or not using tools when needed — are a major failure mode in production ReAct systems.

## Verification Through Execution

The most powerful aspect of tool-augmented reasoning: tool results provide ground-truth feedback.

A model that generates code and runs it knows immediately whether the code works. A model that calls a calculator knows the arithmetic result is correct. This is fundamentally different from pure language reasoning where the model must evaluate its own output.

In production agent systems, this shifts the reliability bottleneck from reasoning accuracy to tool reliability (API availability, timeout handling, error parsing). The reasoning can be verified; tool infrastructure needs to be hardened.